# 02 FreeSurfer recon-all

This notebook runs FreeSurfer reconstruction from standardized MRI inputs.

It does not care whether the T1/T2 input came from `01_convert_mri.ipynb` or was placed directly under `paths.mri_root` beforehand. This supports mixed projects where, for example, one subject already has `T1.mgz` while another subject still needs DICOM conversion.

Supported scenarios:

- T1 only: standard recon-all
- T1 + T2 and `anatomy.recon.use_t2: true`: recon-all with T2 pial refinement
- T1 + T2 and `use_t2: false`: T1-only recon-all
- T2 only: reported as unsupported for this standard workflow

Overwrite is controlled here with `OVERWRITE_STEPS`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import existing_output_policy_for_step, should_overwrite
from meeg_pipeline.anatomy import (
    anatomy_status_to_dataframe,
    discover_mri_subjects,
    resolve_subjects,
    results_to_dataframe,
    run_recon_all_for_subjects,
    write_freesurfer_provenance,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


In [ ]:
SUBJECTS = "all"
OVERWRITE_STEPS = []
DRY_RUN = False

recon_policy = existing_output_policy_for_step("recon", OVERWRITE_STEPS)
provenance_policy = existing_output_policy_for_step("freesurfer_provenance", OVERWRITE_STEPS)

pd.DataFrame([
    {
        "step": "recon",
        "overwrite": should_overwrite("recon", OVERWRITE_STEPS),
        "policy": recon_policy,
    },
    {
        "step": "freesurfer_provenance",
        "overwrite": should_overwrite("freesurfer_provenance", OVERWRITE_STEPS),
        "policy": provenance_policy,
    }
])

In [ ]:
selected_subjects = resolve_subjects(
    SUBJECTS,
    mri_root=config.paths.mri_root,
    subjects_dir=config.freesurfer.subjects_dir,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
)

selected_subjects

## FreeSurfer software provenance

This writes a project-level JSON file documenting the FreeSurfer/MNE/Python environment used by the anatomy workflow.
The file is stored under `derivatives/freesurfer/freesurfer_provenance.json`.


In [ ]:
freesurfer_provenance_result = write_freesurfer_provenance(
    subjects_dir=config.freesurfer.subjects_dir,
    freesurfer_home=config.freesurfer.home,
    on_existing=provenance_policy,
)

results_to_dataframe([freesurfer_provenance_result])


In [ ]:
anatomy_status_to_dataframe(
    selected_subjects,
    subjects_dir=config.freesurfer.subjects_dir,
    mri_root=config.paths.mri_root,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
    spacing=config.anatomy.source_space.spacing,
    bem_ico=config.anatomy.bem.ico,
    bem_conductivity=config.anatomy.bem.conductivity,
)

In [ ]:
recon_results = run_recon_all_for_subjects(
    selected_subjects,
    mri_root=config.paths.mri_root,
    subjects_dir=config.freesurfer.subjects_dir,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
    use_t2=config.anatomy.recon.use_t2,
    freesurfer_home=config.freesurfer.home,
    on_existing=recon_policy,
    dry_run=DRY_RUN,
)

results_to_dataframe(recon_results)

In [ ]:
anatomy_status_to_dataframe(
    selected_subjects,
    subjects_dir=config.freesurfer.subjects_dir,
    mri_root=config.paths.mri_root,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
    spacing=config.anatomy.source_space.spacing,
    bem_ico=config.anatomy.bem.ico,
    bem_conductivity=config.anatomy.bem.conductivity,
)